# Gold publish stream

Purpose: read the Silver GE validated stream, publish business-ready Gold serving rows, maintain a latest-vehicle serving table, and land route-level aggregates plus operational metrics.


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Window
from pyspark.sql import functions as F

CATALOG_NAME = "hant-catalog"
SCHEMA_NAME = "hsl"

STORAGE_ACCOUNT = "streanmingdatasta"
LAKEHOUSE_CONTAINER = "lakehouse"
ROOT_BASE_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/external/hant-catalog"
SILVER_BASE_PATH = f"{ROOT_BASE_PATH}/silver"
GOLD_BASE_PATH = f"{ROOT_BASE_PATH}/gold"

SILVER_VALIDATED_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.silver_vehicle_positions_validated_stream"
SILVER_VALIDATED_PATH_DEFAULT = f"{SILVER_BASE_PATH}/silver_validated_stream/hsl_vehicle_positions"

GOLD_STREAM_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.gold_vehicle_positions_serving_stream"
GOLD_CURRENT_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.gold_vehicle_positions_current"
GOLD_ROUTE_AGG_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.gold_route_direction_5min"
GOLD_PUBLISH_METRICS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.gold_publish_metrics_hsl_vehicle_positions"
GOLD_OPS_EVENTS_TABLE = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.gold_operational_events"

GOLD_STREAM_PATH = f"{GOLD_BASE_PATH}/gold_serving_stream/hsl_vehicle_positions"
GOLD_CURRENT_PATH = f"{GOLD_BASE_PATH}/gold_vehicle_current/hsl_vehicle_positions"
GOLD_ROUTE_AGG_PATH = f"{GOLD_BASE_PATH}/gold_route_direction_5min/hsl_vehicle_positions"
GOLD_PUBLISH_METRICS_PATH = f"{GOLD_BASE_PATH}/gold_metrics/hsl_vehicle_positions_publish"
GOLD_OPS_EVENTS_PATH = f"{GOLD_BASE_PATH}/gold_metrics/operational_events"

CHECKPOINT_BASE = f"{GOLD_BASE_PATH}/checkpoints"
CHECKPOINT_STREAM_PATH = f"{CHECKPOINT_BASE}/gold_vehicle_positions_serving_stream"
CHECKPOINT_CURRENT_PATH = f"{CHECKPOINT_BASE}/gold_vehicle_positions_current"
CHECKPOINT_ROUTE_AGG_PATH = f"{CHECKPOINT_BASE}/gold_route_direction_5min"
CHECKPOINT_METRICS_PATH = f"{CHECKPOINT_BASE}/gold_publish_metrics_hsl_vehicle_positions"

TRIGGER_INTERVAL = "10 seconds"
WATERMARK_DELAY = "15 minutes"


In [0]:
def get_task_value(task_key: str, key: str, debug_value=None):
    try:
        return dbutils.jobs.taskValues.get(taskKey=task_key, key=key, debugValue=debug_value)
    except Exception:
        return debug_value


silver_gate_status = get_task_value("silver_validate_ge", "silver_gate_status", "STREAMING_GE")
silver_gate_reason = get_task_value("silver_validate_ge", "silver_gate_reason", "streaming_silver_ge_not_yet_reported")
silver_validated_path = get_task_value("silver_validate_ge", "silver_validated_path", SILVER_VALIDATED_PATH_DEFAULT)
silver_validation_run_id = get_task_value("silver_validate_ge", "silver_validation_run_id", "streaming_run_pending")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG_NAME}`.{SCHEMA_NAME}")
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_OPS_EVENTS_TABLE} (
    event_ts TIMESTAMP,
    layer STRING,
    upstream_layer STRING,
    upstream_run_id STRING,
    status STRING,
    message STRING,
    source_path STRING
)
USING DELTA
LOCATION "{GOLD_OPS_EVENTS_PATH}"
''')

(
    spark.createDataFrame([{
        "event_ts": None,
        "layer": "gold_publish_stream",
        "upstream_layer": "silver_validate_ge_stream",
        "upstream_run_id": silver_validation_run_id,
        "status": silver_gate_status,
        "message": silver_gate_reason,
        "source_path": silver_validated_path,
    }], schema='''
        event_ts timestamp,
        layer string,
        upstream_layer string,
        upstream_run_id string,
        status string,
        message string,
        source_path string
    ''')
    .withColumn("event_ts", F.current_timestamp())
    .write.format("delta")
    .mode("append")
    .save(GOLD_OPS_EVENTS_PATH)
)


In [0]:
def classify_delay(col_expr):
    return (
        F.when(col_expr.isNull(), F.lit("unknown"))
        .when(col_expr <= F.lit(-60), F.lit("early"))
        .when(col_expr <= F.lit(120), F.lit("on_time"))
        .when(col_expr <= F.lit(300), F.lit("delayed"))
        .otherwise(F.lit("severely_delayed"))
    )


def classify_occupancy(col_expr):
    return (
        F.when(col_expr.isNull(), F.lit("unknown"))
        .when(col_expr <= F.lit(20), F.lit("empty_or_low"))
        .when(col_expr <= F.lit(50), F.lit("moderate"))
        .when(col_expr <= F.lit(80), F.lit("busy"))
        .otherwise(F.lit("crowded"))
    )


try:
    silver_validated_stream_df = spark.readStream.table(SILVER_VALIDATED_TABLE)
except Exception:
    silver_validated_stream_df = spark.readStream.format("delta").load(silver_validated_path)

gold_business_df = (
    silver_validated_stream_df
    .withColumn("canonical_route_id", F.coalesce(F.col("route_id"), F.col("topic_route_id"), F.col("payload_route_id"), F.col("line_id")))
    .withColumn("canonical_direction_id", F.coalesce(F.col("direction_id"), F.col("topic_direction_id"), F.col("dir")))
    .withColumn("gold_publish_ts", F.current_timestamp())
    .withColumn("gold_service_ts", F.coalesce(F.col("event_ts"), F.col("eventhub_enqueued_ts"), F.col("silver_ingest_ts"), F.current_timestamp()))
    .withColumn("service_date", F.coalesce(F.col("operating_day"), F.to_date(F.col("event_ts")), F.col("silver_event_date")))
    .withColumn("service_hour", F.hour(F.col("gold_service_ts")))
    .withColumn("route_direction_key", F.concat_ws("|", F.coalesce(F.col("canonical_route_id"), F.lit("")), F.coalesce(F.col("canonical_direction_id"), F.lit(""))))
    .withColumn("route_vehicle_key", F.concat_ws("|", F.coalesce(F.col("canonical_route_id"), F.lit("")), F.coalesce(F.col("vehicle_id"), F.lit(""))))
    .withColumn("delay_status", classify_delay(F.col("delay_sec")))
    .withColumn("occupancy_status", classify_occupancy(F.col("occupancy")))
    .withColumn(
        "location_quality",
        F.when(F.col("latitude").isNull() | F.col("longitude").isNull(), F.lit("missing"))
         .when((F.col("latitude") < F.lit(59.0)) | (F.col("latitude") > F.lit(61.5)) | (F.col("longitude") < F.lit(23.0)) | (F.col("longitude") > F.lit(26.5)), F.lit("out_of_bounds"))
         .otherwise(F.lit("ok"))
    )
    .withColumn("is_delayed", F.when(F.col("delay_sec").isNull(), F.lit(False)).otherwise(F.col("delay_sec") > F.lit(120)))
    .withColumn("is_severely_delayed", F.when(F.col("delay_sec").isNull(), F.lit(False)).otherwise(F.col("delay_sec") > F.lit(300)))
    .withColumn("has_coordinates", F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
    .withColumn("gold_event_date", F.to_date(F.col("gold_service_ts")))
    .withColumn("event_to_gold_delay_sec", (F.col("gold_publish_ts").cast("long") - F.col("event_ts").cast("long")).cast("long"))
    .withColumn("silver_to_gold_delay_sec", (F.col("gold_publish_ts").cast("long") - F.col("silver_ingest_ts").cast("long")).cast("long"))
    .withColumn("gold_record_type", F.lit("vehicle_position"))
)


In [0]:
def register_table(path: str, table_name: str):
    spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {table_name}
    USING DELTA
    LOCATION "{path}"
    ''')


def precreate_sink(path: str, table_name: str, schema, partition_columns=None):
    if not DeltaTable.isDeltaTable(spark, path):
        writer = (
            spark.createDataFrame([], schema)
            .write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
        )
        if partition_columns:
            writer = writer.partitionBy(*partition_columns)
        writer.save(path)
    register_table(path, table_name)


precreate_sink(GOLD_STREAM_PATH, GOLD_STREAM_TABLE, gold_business_df.schema, partition_columns=["gold_event_date"])
precreate_sink(GOLD_CURRENT_PATH, GOLD_CURRENT_TABLE, gold_business_df.schema, partition_columns=["gold_event_date"])

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_ROUTE_AGG_TABLE} (
    window_start_ts TIMESTAMP,
    window_end_ts TIMESTAMP,
    gold_window_date DATE,
    service_date DATE,
    route_id STRING,
    direction_id STRING,
    line_id STRING,
    transport_mode STRING,
    position_rows BIGINT,
    distinct_vehicle_count BIGINT,
    avg_speed DOUBLE,
    avg_delay_sec DOUBLE,
    max_delay_sec INT,
    avg_occupancy DOUBLE,
    delayed_rows BIGINT,
    delayed_ratio DOUBLE,
    missing_coordinate_rows BIGINT,
    snapshot_ts TIMESTAMP
)
USING DELTA
PARTITIONED BY (gold_window_date)
LOCATION "{GOLD_ROUTE_AGG_PATH}"
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {GOLD_PUBLISH_METRICS_TABLE} (
    batch_id BIGINT,
    batch_ts TIMESTAMP,
    total_rows BIGINT,
    distinct_vehicle_count BIGINT,
    distinct_route_count BIGINT,
    null_route_rows BIGINT,
    null_vehicle_rows BIGINT,
    delayed_rows BIGINT,
    severe_delay_rows BIGINT,
    missing_coordinate_rows BIGINT,
    silver_gate_status STRING,
    silver_validation_run_id STRING
)
USING DELTA
LOCATION "{GOLD_PUBLISH_METRICS_PATH}"
''')


DataFrame[]

In [0]:
def write_gold_metrics(batch_df, batch_id: int):
    if batch_df.isEmpty():
        return

    upstream_validation_run_id = batch_df.agg(
        F.max("silver_validation_run_id").alias("silver_validation_run_id")
    ).collect()[0]["silver_validation_run_id"]

    (
        batch_df.agg(
            F.count("*").alias("total_rows"),
            F.countDistinct("vehicle_id").cast("bigint").alias("distinct_vehicle_count"),
            F.countDistinct("canonical_route_id").cast("bigint").alias("distinct_route_count"),
            F.sum(F.when(F.col("canonical_route_id").isNull() | (F.trim(F.col("canonical_route_id")) == ""), 1).otherwise(0)).cast("bigint").alias("null_route_rows"),
            F.sum(F.when(F.col("vehicle_id").isNull() | (F.trim(F.col("vehicle_id")) == ""), 1).otherwise(0)).cast("bigint").alias("null_vehicle_rows"),
            F.sum(F.when(F.col("is_delayed"), 1).otherwise(0)).cast("bigint").alias("delayed_rows"),
            F.sum(F.when(F.col("is_severely_delayed"), 1).otherwise(0)).cast("bigint").alias("severe_delay_rows"),
            F.sum(F.when(~F.col("has_coordinates"), 1).otherwise(0)).cast("bigint").alias("missing_coordinate_rows"),
        )
        .withColumn("batch_id", F.lit(int(batch_id)).cast("bigint"))
        .withColumn("batch_ts", F.current_timestamp())
        .withColumn("silver_gate_status", F.lit(silver_gate_status))
        .withColumn("silver_validation_run_id", F.lit(upstream_validation_run_id))
        .write.format("delta")
        .mode("append")
        .save(GOLD_PUBLISH_METRICS_PATH)
    )


In [0]:
def upsert_gold_current(batch_df, batch_id: int):
    if batch_df.isEmpty():
        return

    latest_batch_df = (
        batch_df
        .filter(F.col("vehicle_id").isNotNull() & (F.trim(F.col("vehicle_id")) != ""))
        .withColumn(
            "latest_rank",
            F.row_number().over(
                Window.partitionBy("vehicle_id").orderBy(
                    F.col("gold_service_ts").desc_nulls_last(),
                    F.col("gold_publish_ts").desc_nulls_last(),
                    F.col("silver_ingest_ts").desc_nulls_last(),
                )
            )
        )
        .filter(F.col("latest_rank") == 1)
        .drop("latest_rank")
    )

    (
        DeltaTable.forPath(spark, GOLD_CURRENT_PATH)
        .alias("t")
        .merge(latest_batch_df.alias("s"), "t.vehicle_id = s.vehicle_id")
        .whenMatchedUpdateAll(
            condition='''
            coalesce(s.gold_service_ts, s.event_ts, s.eventhub_enqueued_ts, s.gold_publish_ts) >=
            coalesce(t.gold_service_ts, t.event_ts, t.eventhub_enqueued_ts, t.gold_publish_ts)
            '''
        )
        .whenNotMatchedInsertAll()
        .execute()
    )


gold_route_agg_df = (
    gold_business_df
    .withWatermark("gold_service_ts", WATERMARK_DELAY)
    .groupBy(
        F.window("gold_service_ts", "5 minutes"),
        "service_date",
        "canonical_route_id",
        "canonical_direction_id",
        "line_id",
        "transport_mode",
    )
    .agg(
        F.count("*").cast("bigint").alias("position_rows"),
        F.approx_count_distinct("vehicle_id").cast("bigint").alias("distinct_vehicle_count"),
        F.avg("speed").alias("avg_speed"),
        F.avg("delay_sec").alias("avg_delay_sec"),
        F.max("delay_sec").cast("int").alias("max_delay_sec"),
        F.avg("occupancy").alias("avg_occupancy"),
        F.sum(F.when(F.col("is_delayed"), 1).otherwise(0)).cast("bigint").alias("delayed_rows"),
        F.sum(F.when(~F.col("has_coordinates"), 1).otherwise(0)).cast("bigint").alias("missing_coordinate_rows"),
    )
    .select(
        F.col("window.start").alias("window_start_ts"),
        F.col("window.end").alias("window_end_ts"),
        F.to_date(F.col("window.start")).alias("gold_window_date"),
        F.col("service_date"),
        F.col("canonical_route_id").alias("route_id"),
        F.col("canonical_direction_id").alias("direction_id"),
        F.col("line_id"),
        F.col("transport_mode"),
        F.col("position_rows"),
        F.col("distinct_vehicle_count"),
        F.col("avg_speed"),
        F.col("avg_delay_sec"),
        F.col("max_delay_sec"),
        F.col("avg_occupancy"),
        F.col("delayed_rows"),
        (F.col("delayed_rows") / F.col("position_rows").cast("double")).alias("delayed_ratio"),
        F.col("missing_coordinate_rows"),
        F.current_timestamp().alias("snapshot_ts"),
    )
)


In [0]:
for q in spark.streams.active:
    if q.name in {
        "gold_hsl_vehicle_positions_serving_stream",
        "gold_hsl_vehicle_positions_current",
        "gold_hsl_route_direction_5min",
        "gold_hsl_vehicle_positions_publish_metrics",
    }:
        q.stop()

gold_stream_query = (
    gold_business_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_STREAM_PATH)
    .queryName("gold_hsl_vehicle_positions_serving_stream")
    .trigger(processingTime=TRIGGER_INTERVAL)
    .toTable(GOLD_STREAM_TABLE)
)

gold_current_query = (
    gold_business_df.writeStream
    .foreachBatch(upsert_gold_current)
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_CURRENT_PATH)
    .queryName("gold_hsl_vehicle_positions_current")
    .trigger(processingTime=TRIGGER_INTERVAL)
    .start()
)

gold_route_agg_query = (
    gold_route_agg_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_ROUTE_AGG_PATH)
    .queryName("gold_hsl_route_direction_5min")
    .trigger(processingTime=TRIGGER_INTERVAL)
    .toTable(GOLD_ROUTE_AGG_TABLE)
)

gold_metrics_query = (
    gold_business_df.writeStream
    .foreachBatch(write_gold_metrics)
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_METRICS_PATH)
    .queryName("gold_hsl_vehicle_positions_publish_metrics")
    .trigger(processingTime=TRIGGER_INTERVAL)
    .start()
)


In [0]:
for q in spark.streams.active:
    if q.name in {
        "gold_hsl_vehicle_positions_serving_stream",
        "gold_hsl_vehicle_positions_current",
        "gold_hsl_route_direction_5min",
        "gold_hsl_vehicle_positions_publish_metrics",
    }:
        print("NAME:", q.name)
        print("ID:", q.id)
        print("IS ACTIVE:", q.isActive)
        print("STATUS:", q.status)
        print("LAST PROGRESS:", q.lastProgress)
        print("EXCEPTION:", q.exception())
        print("-" * 80)

gold_stream_query.awaitTermination()
gold_current_query.awaitTermination()
gold_route_agg_query.awaitTermination()
gold_metrics_query.awaitTermination()


NAME: gold_hsl_vehicle_positions_publish_metrics
ID: 1975604c-cee1-4365-876f-dce03b8507c3
IS ACTIVE: True
STATUS: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
LAST PROGRESS: None
EXCEPTION: None
--------------------------------------------------------------------------------
NAME: gold_hsl_vehicle_positions_serving_stream
ID: 86a6756d-59bc-4f46-9c41-2cc06de63fa0
IS ACTIVE: True
STATUS: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
LAST PROGRESS: None
EXCEPTION: None
--------------------------------------------------------------------------------
NAME: gold_hsl_vehicle_positions_current
ID: 2462dbc8-0626-44d8-a814-08aa9e6ac60b
IS ACTIVE: True
STATUS: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
LAST PROGRESS: None
EXCEPTION: None
--------------------------------------------------------------------------------
NAME: gold_hsl_route_direction_5min
ID: 8e598b5c-2eed-4272

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can